In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable


# Read only active incremental records from the Control Table
control_df = spark.read.table(
    "control_catalog.metadata.pipeline_control"
)

active_rows = control_df.filter(
    "active = true AND load_type = 'incremental'"
).collect()


# Process each incremental source
for row in active_rows:

    # Get source and target information from the Control Table
    source_path = row["source_path"]

    target_table = (
        f"{row['target_catalog']}."
        f"{row['target_schema']}."
        f"{row['target_table']}"
    )

    primary_key = row["primary_key"]
    watermark_column = row["watermark_column"]
    last_watermark = row["last_watermark"]

    print(f"Reading: {source_path}")
    print(f"Target: {target_table}")
    print(f"Watermark column: {watermark_column}")
    print(f"Last watermark: {last_watermark}")


    # Read the CSV file
    df = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(source_path)


    # Convert the watermark column to timestamp
    df = df.withColumn(
        watermark_column,
        F.col(watermark_column).cast("timestamp")
    )


    # First load: no previous watermark, so load all rows
    if last_watermark is None:

        incremental_df = df

        print("First incremental load: loading all rows.")


    # Later loads: load only rows newer than the last watermark
    else:

        incremental_df = df.filter(
            F.col(watermark_column) > F.lit(last_watermark)
        )

        print("Incremental load: loading only new/changed rows.")


    # Keep only the latest record for each primary key
    window_spec = Window.partitionBy(primary_key).orderBy(
        F.col(watermark_column).desc()
    )

    incremental_df = (
        incremental_df
        .withColumn(
            "row_num",
            F.row_number().over(window_spec)
        )
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )


    # Check whether there is data to process
    if incremental_df.count() == 0:

        print(f"No new data for {target_table}")
        continue


    # Get the latest watermark from the data being processed
    max_watermark = incremental_df.select(
        F.max(watermark_column).alias("max_watermark")
    ).collect()[0]["max_watermark"]


    # First load: create the target Delta table
    if not spark.catalog.tableExists(target_table):

        incremental_df.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_table)

        print(f"Created target table: {target_table}")


    # Later loads: merge new/changed records using the primary key
    else:

        target = DeltaTable.forName(spark, target_table)

        (
            target.alias("t")
            .merge(
                incremental_df.alias("s"),
                f"t.{primary_key} = s.{primary_key}"
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"Merged data into: {target_table}")


    # Update the last processed watermark in the Control Table
    spark.sql(f"""
        UPDATE control_catalog.metadata.pipeline_control
        SET last_watermark = TIMESTAMP '{max_watermark}'
        WHERE source_file = '{row["source_file"]}'
    """)

    print(f"Updated last_watermark: {max_watermark}")
    print(f"Completed: {target_table}")